# Imports

In [1]:
import sys
import time
import numpy as np
from siriuspy.devices import SOFB, HLFOFB, APU, CurrInfoSI
from siriuspy.devices import APU, VPU
import time
from siriuspy.clientconfigdb import ConfigDBClient
import epics
from mathphys.functions import save, load
import h5py
import os
import threading

# Enviroment definitions

In [2]:
os.environ['EPICS_CA_ADDR_LIST'] += ' 10.39.50.33 10.39.50.34'  # Add IPs of EPICS servers of XBPMs

# Class and functions definitions

## XBPMs class

In [3]:
class xbpmMNC:
    def __init__(self, xbpm_nr=1):
        self.prefix = 'SI-09SAFE:DI-PBPM-' + str(xbpm_nr)

    def ampA(self):
        return epics.caget(self.prefix + ':AmplA-Mon'), 0

    def ampB(self):
        return epics.caget(self.prefix + ':AmplB-Mon'), 0

    def ampC(self):
        return epics.caget(self.prefix + ':AmplC-Mon'), 0

    def ampD(self):
        return epics.caget(self.prefix + ':AmplD-Mon'), 0

    def rangeA(self):
        return 0

    def rangeB(self):
        return 0

    def rangeC(self):
        return 0

    def rangeD(self):
        return 0


class xbpmMGN:
    def __init__(self, xbpm_nr=1):
        self.prefix = 'SI-10BCFE:DI-PBPM-' + str(xbpm_nr)

    def ampA(self):
        return epics.caget(self.prefix + ':AmplA-Mon'), 0

    def ampB(self):
        return epics.caget(self.prefix + ':AmplB-Mon'), 0

    def ampC(self):
        return epics.caget(self.prefix + ':AmplC-Mon'), 0

    def ampD(self):
        return epics.caget(self.prefix + ':AmplD-Mon'), 0

    def rangeA(self):
        return 0

    def rangeB(self):
        return 0

    def rangeC(self):
        return 0

    def rangeD(self):
        return 0


class xbpmCAT:
    def __init__(self, xbpm_nr=1):
        self.prefix = 'CAT:FE:PICO01'

    def ampA(self):
        return epics.caget(self.prefix + ':Current1:EngValue'), epics.caget(
            'CAT:FE:PICO01:Current1:EngUnit'
        )

    def ampB(self):
        return epics.caget(self.prefix + ':Current2:EngValue'), epics.caget(
            'CAT:FE:PICO01:Current2:EngUnit'
        )

    def ampC(self):
        return epics.caget(self.prefix + ':Current3:EngValue'), epics.caget(
            'CAT:FE:PICO01:Current3:EngUnit'
        )

    def ampD(self):
        return epics.caget(self.prefix + ':Current4:EngValue'), epics.caget(
            'CAT:FE:PICO01:Current4:EngUnit'
        )

    def rangeA(self):
        return epics.caget('CAT:FE:PICO01:Current1:Range_RBV')

    def rangeB(self):
        return epics.caget('CAT:FE:PICO01:Current2:Range_RBV')

    def rangeC(self):
        return epics.caget('CAT:FE:PICO01:Current3:Range_RBV')

    def rangeD(self):
        return epics.caget('CAT:FE:PICO01:Current4:Range_RBV')


class xbpmCNB:
    def __init__(self, xbpm_nr=1):
        self.prefix = 'CNB:FE:PICO01'

    def ampA(self):
        return epics.caget(self.prefix + ':Current1:EngValue'), epics.caget(
            'CNB:FE:PICO01:Current1:EngUnit'
        )

    def ampB(self):
        return epics.caget(self.prefix + ':Current2:EngValue'), epics.caget(
            'CNB:FE:PICO01:Current2:EngUnit'
        )

    def ampC(self):
        return epics.caget(self.prefix + ':Current3:EngValue'), epics.caget(
            'CNB:FE:PICO01:Current3:EngUnit'
        )

    def ampD(self):
        return epics.caget(self.prefix + ':Current4:EngValue'), epics.caget(
            'CNB:FE:PICO01:Current4:EngUnit'
        )

    def rangeA(self):
        return epics.caget('CNB:FE:PICO01:Current1:Range_RBV')

    def rangeB(self):
        return epics.caget('CNB:FE:PICO01:Current2:Range_RBV')

    def rangeC(self):
        return epics.caget('CNB:FE:PICO01:Current3:Range_RBV')

    def rangeD(self):
        return epics.caget('CNB:FE:PICO01:Current4:Range_RBV')


## Undulators functions

In [4]:
def move_vpu_gap(vpu: VPU, gap, timeout, verbose=False):
    vpu.set_gap(gap)
    time.sleep(0.5)
    print('Gap-RB {:.3f} mm'.format(vpu.gap)) if verbose else 0
    if vpu.cmd_move_start(timeout):
        time.sleep(0.5)
        print('Undulator is moving...') if verbose else 0
        while vpu.is_moving:
            time.sleep(0.1)
            print('Current gap {:.3f} mm.'.format(vpu.gap_mon), end='\r') if verbose else 0
        print('Gap {:.3f} mm reached.'.format(vpu.gap)) if verbose else 0
        return True
    else:
        print('Error while cmd_move_start.')
        return False

def move_apu_phase(apu: APU, phase, timeout, verbose=False):
    apu.set_phase(phase)
    time.sleep(0.5)
    print('phase-RB {:.3f} mm'.format(apu.phase)) if verbose else 0
    if apu.cmd_move_start(timeout):
        time.sleep(0.5)
        print('Undulator is moving...') if verbose else 0
        while apu.is_moving:
            time.sleep(0.1)
            print('Current phase {:.3f} mm.'.format(apu.phase_mon), end='\r') if verbose else 0
        print('phase {:.3f} mm reached.'.format(apu.phase)) if verbose else 0
        return True
    else:
        print('Error while cmd_move_start.')
        return False

## Measurement algorithm functions

In [ ]:
def meas(
    idlist,
    xbpmlist,
    names,
    sofb: SOFB,
    currinfo: CurrInfoSI,
    subsec='09SA',
    agx=0,
    agy=0,
    psx=0,
    psy=0,
    nr_meas=10,
    tinterval=1,
):
    blades = dict()
    for i, xbpm in enumerate(xbpmlist):
        blades[names[i]] = {
            'prefix': xbpm.prefix,
            'A_val': [],
            'B_val': [],
            'C_val': [],
            'D_val': [],
            'A_range': [],
            'B_range': [],
            'C_range': [],
            'D_range': [],
        }

    for j in np.arange(nr_meas):
        for i, xbpm in enumerate(xbpmlist):
            xbpm_ampA = xbpm.ampA()
            xbpm_ampB = xbpm.ampB()
            xbpm_ampC = xbpm.ampC()
            xbpm_ampD = xbpm.ampD()
            xbpm_rangeA = xbpm.rangeA()
            xbpm_rangeB = xbpm.rangeB()
            xbpm_rangeC = xbpm.rangeC()
            xbpm_rangeD = xbpm.rangeD()

            blades[names[i]]['A_val'].append(xbpm_ampA)
            blades[names[i]]['B_val'].append(xbpm_ampB)
            blades[names[i]]['C_val'].append(xbpm_ampC)
            blades[names[i]]['D_val'].append(xbpm_ampD)
            blades[names[i]]['A_range'].append(xbpm_rangeA)
            blades[names[i]]['B_range'].append(xbpm_rangeB)
            blades[names[i]]['C_range'].append(xbpm_rangeC)
            blades[names[i]]['D_range'].append(xbpm_rangeD)
        time.sleep(tinterval)

    vpu_cnb = idlist[0]
    apu_mnc = idlist[1]

    idinfo = {
        'cnb': vpu_cnb.gap_mon,
        'mnc': apu_mnc.phase_mon,
    }

    machinfo = {
        'current': currinfo.current,
        'agx': agx,
        'agy': agy,
        'posx': psx,
        'posy': psy,
        'orbx': sofb.orbx,
        'orby': sofb.orby,
    }

    data = (blades, idinfo, machinfo)

    fname = 'xbpm_acq_subsec:'
    fname += f'{subsec}_agx:{agx:03.0f}_agy:{agy:03.0f}_'
    fname += f'posx:{psx:03.0f}_posy:{psy:03.0f}_'
    fname += 'time_' + str(time.time()) + '.pickle'
    save(data, fname)

    return data


def implement_bump(
    sofb,
    subsec='09SA',
    refx=None,
    refy=None,
    agx=0,
    agy=0,
    psx=0,
    psy=0,
    tol_orb=3,
):
    def get_rms(refx, refy, idx):
        dorbx = sofb.orbx - refx
        dorby = sofb.orby - refy
        dorbx = dorbx[idx]
        dorby = dorby[idx]
        return np.hstack([dorbx, dorby]).std()

    max_tol_orb = 10

    # Get ref orb
    if refx is None or refy is None:
        clt = ConfigDBClient(config_type='si_orbit')
        ref_orb = clt.get_config_value('ref_orb')
        refx = np.array(ref_orb['x'])
        refy = np.array(ref_orb['y'])

    # Calculate bumps
    orbx, orby = sofb.si_calculate_bumps(
        refx, refy, subsec=subsec, agx=agx, agy=agy, psx=psx, psy=psy
    )

    # Find out where bump is being made
    dummy, _ = sofb.si_calculate_bumps(refx, refy, subsec=subsec, agx=10)
    idx = ~np.isclose(dummy, refx)
    strt = idx.nonzero()[0][0]

    # Set orbit on SOFB
    sofb.refx = orbx
    sofb.refy = orby
    enbl = np.ones(orbx.size, dtype=bool)
    nr_bpms = 4
    enbl[strt - nr_bpms : strt] = False
    enbl[strt + 2 : strt + 2 + nr_bpms] = False
    sofb.bpmxenbl = enbl
    sofb.bpmyenbl = enbl

    # Verify orbit correction
    rms_residue = tol_orb + 1
    nr_iters = 10
    print('Waiting orbit...')
    while rms_residue > tol_orb:
        _ = sofb.correct_orbit_manually(nr_iters=nr_iters, residue=1)
        rms_residue = get_rms(orbx, orby, idx)
        print(f'    rms_residue = {rms_residue:.3f} um')
        tol_orb *= 1.2
        if tol_orb > max_tol_orb:
            raise ValueError('Could not correct orbit.')

    print('Done!')


def restore_sofb_reforb(sofb, bpmxenbl, bpmyenbl):
    clt = ConfigDBClient(config_type='si_orbit')
    ref_orb = clt.get_config_value('ref_orb')
    refx = np.array(ref_orb['x'])
    refy = np.array(ref_orb['y'])
    sofb.refx = refx
    sofb.refy = refy
    sofb.bpmxenbl = bpmxenbl
    sofb.bpmyenbl = bpmyenbl


def is_beam_alive(currinfo, sofb, idlist):
    if currinfo.storedbeam:
        return True
    print('Beam is dead!')
    restore_sofb_reforb(sofb)
    move_apu_phase(idlist[1], 11)
    move_vpu_gap(idlist[0], 80)
    return False


def stop_meas_func(sofb, idlist):
    if not stop_meas:
        return False
    restore_sofb_reforb(sofb)
    sofb.correct_orbit_manually(10, 2)
    move_apu_phase(idlist[1], 11)
    move_vpu_gap(idlist[0], 80)
    return True


def do_bumps(angsx, angsy, subsec, idlist, currinfo, sofb, xbpmlist, names):
    break_loop = False
    for i, agx in enumerate(angsx):
        if break_loop:
            break

        for j, agy in enumerate(angsy):
            if not is_beam_alive(currinfo, sofb, idlist) or stop_meas_func(
                sofb, idlist
            ):
                break_loop = True
                break

            if i % 2:
                agy *= -1
            print(f'agx:{agx:.0f}    agy:{agy:.0f}')
            try:
                implement_bump(sofb=sofb, subsec=subsec, agx=agx, agy=agy)
                meas(
                    idlist,
                    xbpmlist,
                    names,
                    sofb,
                    currinfo,
                    subsec=subsec,
                    agx=agx,
                    agy=agy,
                    nr_meas=10,
                    tinterval=1,
                )
                time.sleep(0.1)
                while pause_meas:
                    time.sleep(2)
                    if stop_meas:
                        break
                    print(
                        'Paused, set pause_meas to False to continue....',
                        end='\r',
                    )
            except ValueError as err:
                print(err)
                continue
    restore_sofb_reforb(sofb)

# Measurement procedure

## Define devices

In [5]:
sofb = SOFB(SOFB.DEVICES.SI)
currinfo = CurrInfoSI()
clt = ConfigDBClient(config_type="si_orbit")
ref_orb = clt.get_config_value("ref_orb")
refx = np.array(ref_orb["x"])
refy = np.array(ref_orb["y"])

apu = APU(APU.DEVICES.APU22_09SA)
vpu = VPU(VPU.DEVICES.VPU29_06SB)

In [8]:
xbpm_cnb = xbpmCNB(1)
xbpm_mnc1 = xbpmMNC(1)
xbpm_mnc2 = xbpmMNC(2)
xbpm_mgn1 = xbpmMGN(1)
xbpm_mgn2 = xbpmMGN(2)

idlist = [vpu, apu]

## Turn-off SOFB correction loop and get initial BPMs states

In [ ]:
sofb.cmd_turn_off_autocorr()
bpmxenbl = sofb.bpmxenbl
bpmyenbl = sofb.bpmyenbl

## Do BUMPS @ MANACA

In [ ]:
# Configure phase
phase = 0
move_apu_phase(apu, phase, 30)

# Configure grid
npoints = 11
angsx = np.linspace(-20, 20, npoints)
angsy = np.linspace(-20, 20, npoints)

# Select appropriate section and start measurement
stop_meas = False
pause_meas = False
subsec = '09SA'
xbpmlist = [xbpm_mnc1, xbpm_mnc2]
names = ['MNC1', 'MNC2']
t1 = threading.Thread(target=do_bumps, args=(angsx, angsy, subsec, idlist))
t1.start()

# Do bumps @ MOGNO

In [ ]:
# Configure grid
angsx = np.array([0])
angsy = np.linspace(-20, 20, 11)

# Select appropriate section and start measurement
stop_meas = False
pause_meas = False
subsec = '10BC'
xbpmlist = [xbpm_mgn1, xbpm_mgn2]
names = ['MGN1', 'MGN2']
t1 = threading.Thread(target=do_bumps, args=(angsx, angsy, subsec, idlist))
t1.start()

# Do bumps @ CARNAUBA

In [ ]:
# Configure gap
gap = 0
move_vpu_gap(vpu, gap)

# Configure grid
angsx = np.linspace(-20, 20, 11)
angsy = np.linspace(-20, 20, 11)

# Select appropriate section and start measurement
stop_meas = False
pause_meas = False
subsec = '06SB'
xbpmlist = [xbpm_cnb]
names = ['CNB']
t1 = threading.Thread(target=do_bumps, args=(angsx, angsy, subsec, idlist))
t1.start()

In [14]:
restore_sofb_reforb(sofb, bpmxenbl, bpmyenbl)